# Preparación de datos

**Conjunto de datos:** Dataset 2 - Lugares de emisiones

**Nombre de archivo:** emission_permits_anom_2.json

## 0. Inicialización

Instalar ydata-profiling

In [434]:
!pip install ydata-profiling

Importaciones

In [435]:
import pandas as pd
import matplotlib.pyplot as plt
from ydata_profiling import ProfileReport
import seaborn as sns

import json
from tabulate import tabulate

Visualización de tablas y gráficas

In [436]:
sns.set_style("darkgrid")

def print_table(df):
    print(tabulate(df, headers='keys', tablefmt='simple_outline'))

Lectura y muestra del archivo

In [437]:
# 1. Cargar el JSON
with open('../data/original/emission_permits_anom_2.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

# 2. Extraer features
features = data['features']

# 3. Construir DataFrame con properties y coordenadas
df = pd.DataFrame([
    {
        **feature['properties'],
        'Latitud': feature['geometry']['coordinates'][1],
        'Longitud': feature['geometry']['coordinates'][0]
    }
    for feature in features
])

df.head()

,IDExpediente,Estado,Regional,Departamento,Municipio,Vereda,Class,TipoCombustible,TipoFuenteEmision,Cuenca,Latitud,Longitud
0,73640.0,Seguimiento y Control,Sabana Occidente,Cundinamarca,MOSQUERA,CENTRO,None,Otros,Horno,Río Bogotá,4.703418,-74.226561
1,73788.0,Seguimiento y Control,Ubate,Cundinamarca,LENGUAZAQUE,Resguardo,None,Carbón,Caldera Horno,Río Suárez,5.318407,-73.704281
2,74314.0,Seguimiento y Control,Sabana Occidente,Cundinamarca,MADRID,LA PUNTA,None,ACPM,Caldera Horno,Río Bogotá,4.800462,-74.210355
3,75972.0,Seguimiento y Control,Sabana Occidente,Cundinamarca,MOSQUERA,Balsillas,None,Fuel Oil No.8,Planta de Asfalto,Río Bogotá,4.678797,-74.284112
4,78824.0,Seguimiento y Control,Sabana Occidente,Cundinamarca,FUNZA,El Hato,None,Carbón,Caldera Horno,Río Bogotá,4.699590,-74.193752


**1.** Transformar IDExpediente a integer

In [438]:
df['IDExpediente'] = df['IDExpediente'].astype("Int64")

**2.** Eliminar columnas innecesarias

In [439]:
df = df.drop(columns=['Cuenca'])

**3.** Eliminar duplicados

In [440]:
df = df.drop_duplicates()

**4.** Eliminar empresas con localización inválida

In [441]:
df = df[(df['Latitud'] < 12.27) & (df['Longitud'] < -66.50)]

**5.** Manejar la capitalización en las variables categóricas

In [442]:
df["Vereda"] = df["Vereda"].str.strip().str.upper()
df["TipoFuenteEmision"] = df["TipoFuenteEmision"].str.strip().str.capitalize()
df.head()

,IDExpediente,Estado,Regional,Departamento,Municipio,Vereda,Class,TipoCombustible,TipoFuenteEmision,Latitud,Longitud
0,73640,Seguimiento y Control,Sabana Occidente,Cundinamarca,MOSQUERA,CENTRO,None,Otros,Horno,4.703418,-74.226561
1,73788,Seguimiento y Control,Ubate,Cundinamarca,LENGUAZAQUE,RESGUARDO,None,Carbón,Caldera horno,5.318407,-73.704281
2,74314,Seguimiento y Control,Sabana Occidente,Cundinamarca,MADRID,LA PUNTA,None,ACPM,Caldera horno,4.800462,-74.210355
3,75972,Seguimiento y Control,Sabana Occidente,Cundinamarca,MOSQUERA,BALSILLAS,None,Fuel Oil No.8,Planta de asfalto,4.678797,-74.284112
4,78824,Seguimiento y Control,Sabana Occidente,Cundinamarca,FUNZA,EL HATO,None,Carbón,Caldera horno,4.699590,-74.193752


**6.** Mejorar la presentación de los (sin definir)

In [443]:
df["TipoCombustible"] = df["TipoCombustible"].replace("(sin definir)", "Sin definir")
df["TipoFuenteEmision"] = df["TipoFuenteEmision"].replace("(sin definir)", "Sin definir")
df[(df["TipoCombustible"] == "Sin definir") | (df["TipoFuenteEmision"] == "Sin definir")]

,IDExpediente,Estado,Regional,Departamento,Municipio,Vereda,Class,TipoCombustible,TipoFuenteEmision,Latitud,Longitud
42,138622,Seguimiento y Control,Sabana Centro,Cundinamarca,NEMOCON,PATIO BONITO,None,Sin definir,Sin definir,5.118908,-73.895435
43,139320,Seguimiento y Control,Ubate,Cundinamarca,TAUSA,RASGATÁ,None,Sin definir,Horno,5.190613,-73.879585
61,150306,Seguimiento y Control,Sabana Centro,Cundinamarca,NEMOCON,PATIO BONITO,None,Sin definir,Sin definir,5.125421,-73.904398
100,171196,Seguimiento y Control,Bogotá y Municipio de la Calera,Distrito Capital,LOCALIDAD DE CIUDAD BOLIVAR,MOCHUELO BAJO,None,Sin definir,Sin definir,4.506914,-74.150135
102,171204,Seguimiento y Control,Bogotá y Municipio de la Calera,Distrito Capital,LOCALIDAD DE CIUDAD BOLIVAR,MOCHUELO BAJO,None,Sin definir,Sin definir,4.516030,-74.149127
...,...,...,...,...,...,...,...,...,...,...,...
533,284322,Sancionatorio,Chiquinquira,Boyacá,CHIQUINQUIRA,SUCRE ORIENTAL,Por emisiones atmosféricas sin permiso o no cu...,Sin definir,Sin definir,5.558346,-73.823302
534,287776,Sancionatorio,Ubate,Cundinamarca,UBATE,CASCO URBANO,Por emisiones atmosféricas sin cumplir con los...,Sin definir,Sin definir,5.297332,-73.816230
538,290848,Sancionatorio,Ubate,Cundinamarca,TAUSA,RASGATÁ,Por emisiones atmosféricas sin permiso o no cu...,Sin definir,Sin definir,5.186102,-73.883708
540,291712,Sancionatorio,Sumapaz,Cundinamarca,ARBELAEZ,SAN ROQUE,Por emisiones atmosféricas sin cumplir con los...,Sin definir,Sin definir,4.274013,-74.446186


**7.** Estandarizar los tipos de fuente

In [444]:
df["TipoFuenteEmision"].value_counts()

TipoFuenteEmision
Sin definir                       273
Horno                             117
Caldera                            21
Caldera horno                      14
Secadores                           7
Planta de asfalto                   6
Molino                              3
Chimenea 1                          3
Noaplica (área de operación)        2
Trituradora                         2
Planta de asfalto adm               1
Aspiración molino danioni i         1
Barrilado de grafito                1
Campana de extracción  plomo 1      1
Triturador de escombros             1
Triturador de material              1
Horno túnel 1 soacha 2              1
Chimenea triunfo central            1
Planta de mezcla asfáltica          1
Batería de coquización a            1
Molino buhler                       1
Planta trituradora                  1
Batería de producción de coque      1
Name: count, dtype: int64

In [445]:
df["TipoFuenteEmision"] = df["TipoFuenteEmision"].replace({
    "Chimenea 1": "Chimenea",
    "Noaplica (área de operación)": "No aplica (área de operación)",
    "Planta de asfalto adm": "Planta de asfalto",
    "Barrilado de grafito": "Horno",
    "Filtro molino pendular": "Molino",
    "Aspiración molino danioni i": "Molino",
    "Triturador de escombros": "Trituradora",
    "Campana de extracción  plomo 1": "Horno",
    "Horno de secado": "Horno",
    "Horno arcillas de soacha tipo túnel": "Horno",
    "Triturador de material": "Trituradora",
    "Horno túnel 1 soacha 2": "Horno",
    "Chimenea triunfo central": "Chimenea",
    "700 bhp vr2": "Caldera",
    "Planta de mezcla asfáltica": "Planta de asfalto",
    "Batería de coquización a": "Batería de coquización",
    "Molino buhler": "Molino",
    "Planta trituradora": "Trituradora",
    "Batería de producción de coque": "Batería de coquización",
})

In [446]:
df["TipoFuenteEmision"].value_counts()

TipoFuenteEmision
Sin definir                      273
Horno                            120
Caldera                           21
Caldera horno                     14
Planta de asfalto                  8
Secadores                          7
Molino                             5
Trituradora                        5
Chimenea                           4
No aplica (área de operación)      2
Batería de coquización             2
Name: count, dtype: int64

**8.** Estandarizar los tipos de combustible

In [447]:
df["TipoCombustible"].value_counts()

TipoCombustible
Sin definir      274
Carbón           115
Otros             25
Gas               16
ACPM              13
NoAplica           6
Coque              5
Fuel Oil No.8      4
Mezcla             2
Madera             1
Name: count, dtype: int64

In [448]:
df["TipoCombustible"] = df["TipoCombustible"].replace("NoAplica", "No aplica")

In [449]:
df["TipoCombustible"].value_counts()

TipoCombustible
Sin definir      274
Carbón           115
Otros             25
Gas               16
ACPM              13
No aplica          6
Coque              5
Fuel Oil No.8      4
Mezcla             2
Madera             1
Name: count, dtype: int64

**9.** Estandarizar los municipios

In [450]:
df["Municipio"] = df["Municipio"].replace("LOC.USAQUEN CERROS ORIENTALES", "LOCALIDAD DE USAQUEN")

In [451]:
print_table(pd.DataFrame(df["Municipio"].value_counts()))

┌─────────────────────────────┬─────────┐
│ Municipio                   │   count │
├─────────────────────────────┼─────────┤
│ LOCALIDAD DE CIUDAD BOLIVAR │      42 │
│ NEMOCON                     │      39 │
│ SOACHA                      │      34 │
│ COGUA                       │      33 │
│ GIRARDOT                    │      23 │
│ CUCUNUBA                    │      18 │
│ TAUSA                       │      17 │
│ MOSQUERA                    │      14 │
│ LENGUAZAQUE                 │      14 │
│ GUACHETA                    │      14 │
│ CAJICA                      │      13 │
│ FUSAGASUGA                  │      12 │
│ SUTATAUSA                   │      11 │
│ VILLAPINZON                 │      11 │
│ TOCANCIPA                   │      11 │
│ COTA                        │      10 │
│ ZIPAQUIRA                   │      10 │
│ FACATATIVA                  │       9 │
│ RAQUIRA                     │       9 │
│ MADRID                      │       7 │
│ UBATE                       │   

**10.** Obtener probabilidades de incidencia de las empresas

In [452]:
fuel_matrix = pd.read_csv('../data/enriquecida/air_fuel_matrix.csv')
source_matrix = pd.read_csv('../data/enriquecida/air_source_matrix.csv')

In [453]:
# Nos aseguramos de que los nombres de columnas coincidan
fuel_matrix = fuel_matrix.rename(columns={'Tipo de combustible': 'TipoCombustible'})
source_matrix = source_matrix.rename(columns={'Tipo de fuente': 'TipoFuenteEmision'})

for var in fuel_matrix['Variable'].unique():
    # Subconjuntos para esta variable específica
    fuel_sub = (
        fuel_matrix[fuel_matrix['Variable'] == var]
        .rename(columns={
            'Probabilidad': f'Probabilidad_Fuel_{var}',
            'Ponderación': f'Ponderacion_Fuel_{var}'
        })[['TipoCombustible', f'Probabilidad_Fuel_{var}', f'Ponderacion_Fuel_{var}']]
    )
    
    src_sub = (
        source_matrix[source_matrix['Variable'] == var]
        .rename(columns={
            'Probabilidad': f'Probabilidad_Source_{var}',
            'Ponderación': f'Ponderacion_Source_{var}'
        })[['TipoFuenteEmision', f'Probabilidad_Source_{var}', f'Ponderacion_Source_{var}']]
    )
    
    # Merges sobre df
    df = df.merge(fuel_sub, on='TipoCombustible', how='left')
    df = df.merge(src_sub, on='TipoFuenteEmision', how='left')
    
    # Calcular probabilidad
    df[f'ProbabilidadIncidencia_{var}'] = (
        (df[f'Probabilidad_Fuel_{var}'] * df[f'Ponderacion_Fuel_{var}'] +
         df[f'Probabilidad_Source_{var}'] * df[f'Ponderacion_Source_{var}']) / 9
    )
    
    # Eliminar las columnas intermedias
    df = df.drop(columns=[
        f'Probabilidad_Fuel_{var}', 
        f'Ponderacion_Fuel_{var}',
        f'Probabilidad_Source_{var}', 
        f'Ponderacion_Source_{var}'
    ])

Resultado final

In [454]:
df

,IDExpediente,Estado,Regional,Departamento,Municipio,Vereda,Class,TipoCombustible,TipoFuenteEmision,Latitud,...,ProbabilidadIncidencia_NO2,ProbabilidadIncidencia_CO,ProbabilidadIncidencia_Temp,ProbabilidadIncidencia_NO,ProbabilidadIncidencia_PM10,ProbabilidadIncidencia_BP,ProbabilidadIncidencia_NOX,ProbabilidadIncidencia_O3,ProbabilidadIncidencia_RAIN,ProbabilidadIncidencia_SRAD
0,73640,Seguimiento y Control,Sabana Occidente,Cundinamarca,MOSQUERA,CENTRO,None,Otros,Horno,4.703418,...,0.711111,0.577778,0.055556,0.711111,0.800000,0.0,0.755556,0.333333,0.055556,0.055556
1,73788,Seguimiento y Control,Ubate,Cundinamarca,LENGUAZAQUE,RESGUARDO,None,Carbón,Caldera horno,5.318407,...,0.688889,0.644444,0.055556,0.688889,0.800000,0.0,0.733333,0.266667,0.055556,0.166667
2,74314,Seguimiento y Control,Sabana Occidente,Cundinamarca,MADRID,LA PUNTA,None,ACPM,Caldera horno,4.800462,...,0.888889,0.644444,0.000000,0.888889,0.800000,0.0,0.933333,0.400000,0.000000,0.111111
3,75972,Seguimiento y Control,Sabana Occidente,Cundinamarca,MOSQUERA,BALSILLAS,None,Fuel Oil No.8,Planta de asfalto,4.678797,...,0.777778,0.600000,0.055556,0.777778,0.844444,0.0,0.822222,0.444444,0.055556,0.166667
4,78824,Seguimiento y Control,Sabana Occidente,Cundinamarca,FUNZA,EL HATO,None,Carbón,Caldera horno,4.699590,...,0.688889,0.644444,0.055556,0.688889,0.800000,0.0,0.733333,0.266667,0.055556,0.166667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
456,284322,Sancionatorio,Chiquinquira,Boyacá,CHIQUINQUIRA,SUCRE ORIENTAL,Por emisiones atmosféricas sin permiso o no cu...,Sin definir,Sin definir,5.558346,...,0.266667,0.266667,0.055556,0.266667,0.422222,0.0,0.311111,0.133333,0.055556,0.055556
457,287776,Sancionatorio,Ubate,Cundinamarca,UBATE,CASCO URBANO,Por emisiones atmosféricas sin cumplir con los...,Sin definir,Sin definir,5.297332,...,0.266667,0.266667,0.055556,0.266667,0.422222,0.0,0.311111,0.133333,0.055556,0.055556
458,290848,Sancionatorio,Ubate,Cundinamarca,TAUSA,RASGATÁ,Por emisiones atmosféricas sin permiso o no cu...,Sin definir,Sin definir,5.186102,...,0.266667,0.266667,0.055556,0.266667,0.422222,0.0,0.311111,0.133333,0.055556,0.055556
459,291712,Sancionatorio,Sumapaz,Cundinamarca,ARBELAEZ,SAN ROQUE,Por emisiones atmosféricas sin cumplir con los...,Sin definir,Sin definir,4.274013,...,0.266667,0.266667,0.055556,0.266667,0.422222,0.0,0.311111,0.133333,0.055556,0.055556


In [455]:
reporte = ProfileReport(df)
reporte.to_file("archivos_generados/Reporte perfilamiento - Dataset 2 Final.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 26/26 [00:01<00:00, 22.59it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Exportar a CSV

In [457]:
df.to_csv('../data/preparada/emission_permits.csv', index=False)